# Conductance-LIF Pipeline — nc_sim Topology, Noise-Only, Stimulus-Driven

nc_sim spatial axon-growth topology, noise-only background drive (no tonic driver neurons), and periodic cluster stimulation with short-term synaptic depression.

In [ ]:
import importlib
import inspect
import random

import matplotlib.pyplot as plt
import numpy as np

from lif_simulation import analysis as analysis_module
from lif_simulation import models as models_module
from lif_simulation import network as network_module
from lif_simulation import plotting as plotting_module
from lif_simulation import probes as probes_module
from lif_simulation import session_io as session_io_module
from lif_simulation import session_views as session_views_module
from lif_simulation import simulation as simulation_module
from lif_simulation import stimulation as stimulation_module
from lif_simulation import voltage_storage as voltage_storage_module
from lif_simulation import workflows as workflows_module

for module in (
    models_module,
    network_module,
    simulation_module,
    stimulation_module,
    voltage_storage_module,
    session_io_module,
    analysis_module,
    probes_module,
    workflows_module,
    plotting_module,
    session_views_module,
):
    importlib.reload(module)

analyze_spike_trains = analysis_module.analyze_spike_trains
compute_hub_firing_rate_groups = analysis_module.compute_hub_firing_rate_groups
validate_hub_structure = analysis_module.validate_hub_structure

plot_combined_network_layout = plotting_module.plot_combined_network_layout
plot_firing_rates = plotting_module.plot_firing_rates
plot_hub_degree_distributions = plotting_module.plot_hub_degree_distributions
plot_hub_firing_rate_histogram = plotting_module.plot_hub_firing_rate_histogram
plot_hub_network = plotting_module.plot_hub_network
plot_network_positions = plotting_module.plot_network_positions
plot_raster = plotting_module.plot_raster
plot_resampled_raster = plotting_module.plot_resampled_raster
plot_sag_probe = plotting_module.plot_sag_probe
plot_spike_train_analysis_summary = plotting_module.plot_spike_train_analysis_summary
plot_voltage_heatmap = plotting_module.plot_voltage_heatmap
plot_voltage_traces = plotting_module.plot_voltage_traces

run_h_current_step_probe = probes_module.run_h_current_step_probe
run_no_stimulation_validation = probes_module.run_no_stimulation_validation
summarize_h_current_step_probe = probes_module.summarize_h_current_step_probe
load_session_bundle = session_views_module.load_session_bundle
sequential_simulation_individual_saves = workflows_module.sequential_simulation_individual_saves

print(f"Using workflow module: {workflows_module.__file__}")
print(inspect.signature(sequential_simulation_individual_saves))

In [ ]:
seed = 41
np.random.seed(seed)
random.seed(seed)

# Dispatch trace (verified against lif_simulation/workflows.py):
#   network_source == 'nc_sim'   -> network built by build_network_from_nc_sim; the
#                                   clustered-builder params below are IGNORED, and no hubs are made.
#   stimulation_enabled == True  -> cluster_fraction>0 AND neurons_per_cluster>0 AND
#                                   0 < burst_interval <= recording_duration. In stim mode the
#                                   spontaneous_* params are IGNORED (no frozen baseline drive is
#                                   assigned; LIFNeuron.i_baseline stays 0.0), so the ONLY stochastic
#                                   drive between stimuli is background_noise_sigma.
# Tags: [nc_sim]=used by nc_sim builder  [stim]=used for stimulation  [always]=used regardless
#       [clustered-only:IGNORED here]  [spontaneous-only:IGNORED in stim mode]
simulation_params = {
    # --- run / dataset ---
    'n_recordings': 1,                              # [always] recordings from the same network; raise for a multi-recording inference dataset
    'recording_duration': 60000,                    # [always] ms; >> burst_interval -> ~8 stim-evoked bursts (60000/7000)
    'save_dir': 'LIF data',                         # [always] session output root: LIF data/<timestamp>/
    'dt': 0.1,                                      # [always] integration step (ms); also the saved raw-voltage step

    # --- topology: nc_sim grown spatial network ---
    'network_source': 'nc_sim',                     # [dispatch] use build_network_from_nc_sim instead of create_clustered_network
    'nc_sim_width': 2.0,                            # [nc_sim] culture width (mm)
    'nc_sim_height': 2.0,                           # [nc_sim] culture height (mm); rho*W*H = 150*2*2 ~ 600 neurons
    'nc_sim_rho': 150.0,                            # [nc_sim] neuron density (neurons/mm^2)
    'nc_sim_axon_length': 1.0,                      # [nc_sim] mean axon length (mm); sets connection range / burst synchrony
    'nc_sim_obstacles': None,                       # [nc_sim] obstacle grid H (None = flat isotropic culture)
    'num_clusters': 20,                             # [nc_sim + stim] spatial partition for weight sampling AND for stim cluster targeting

    # --- NOISE-ONLY drive ---
    'background_noise_sigma': 0.5,                  # [used] per-neuron membrane-noise std; in stim mode this is the ONLY stochastic drive
    'spontaneous_baseline_mean': 0.0,              # [spontaneous-only:IGNORED in stim] frozen baseline drive mean (drivers OFF; 0 keeps it noise-only if switched to spontaneous)
    'spontaneous_baseline_sd': 0.0,                # [spontaneous-only:IGNORED in stim] frozen baseline drive std
    'spontaneous_baseline_seed': seed,             # [spontaneous-only:IGNORED in stim] baseline-draw seed (also overwritten per-session in the run cell)
    'spontaneous_baseline_distribution': 'lognormal',  # [spontaneous-only:IGNORED in stim] baseline-draw distribution
    'spontaneous_noise_sigma': 0.0,                # [spontaneous-only:IGNORED in stim] membrane noise used INSTEAD of background_noise_sigma in spontaneous mode
    'spontaneous_adaptation_tau_scale': 3.0,       # [spontaneous-only:IGNORED in stim] adaptation tau scaling
    'spontaneous_adaptation_increment_scale': 1.0, # [spontaneous-only:IGNORED in stim] adaptation increment scaling
    'spontaneous_exc_weight_scale': 0.65,          # [spontaneous-only:IGNORED in stim] recurrent-exc weight scaling (stim mode uses 1.0)
    'spontaneous_burst_frac_thresh': 0.12,         # [spontaneous-only:IGNORED in stim] pop-rate burst-detection threshold (segment_states runs only in spontaneous mode)

    # --- STIMULATION (ENABLED here) ---
    'burst_interval': 7000,                         # [stim] ms between stim events
    'burst_interval_jitter': 1500,                  # [stim] +/- jitter applied to each stim onset (ms)
    'cluster_fraction': 0.7,                        # [stim] fraction of spatial clusters stimulated per event (>0 => stim enabled)
    'neurons_per_cluster': 6,                       # [stim] neurons stimulated per chosen cluster (>0 => stim enabled)
    'stim_amplitude_range': (2.0, 3.5),             # [stim] per-stimulus amplitude range (nA)
    'stim_duration_range': (10, 30),                # [stim] per-stimulus duration range (ms)

    # --- short-term synaptic depression (self-terminating bursts) ---
    'depressing': True,                             # [nc_sim/used] build DepressingExpSynapse so recurrent bursts self-terminate
    'tau_q': 4000.0,                                # [used iff depressing] resource-recovery time constant (ms)
    'delta_q': 0.8,                                 # [used iff depressing] per-spike resource-depletion fraction

    # --- membrane / recording ---
    'use_h_current': True,                          # [nc_sim/used] enable the slow h-current on every neuron
    'record_voltage': True,                         # [always] write raw full-dt voltage (set False to skip HDF5 and run much faster)
    'voltage_sample_rate': 0.1,                     # [legacy/metadata] requested sample interval; saved traces are raw full-dt (= dt)
    'voltage_storage_backend': 'hdf5_external',     # [used iff record_voltage] chunked external HDF5 sidecars (needs h5py)
    'voltage_chunk_samples': 4096,                  # [used iff record_voltage] HDF5 flush chunk size (samples)

    # --- clustered-builder params: IGNORED when network_source='nc_sim' ---
    'space_size': 15,                               # [clustered-only:IGNORED] square embedding size for create_clustered_network
    'max_connection_distance': 6.0,                 # [clustered-only:IGNORED] max connection distance for create_clustered_network
    'neurons_per_cluster_range': (12, 18),          # [clustered-only:IGNORED] cluster-size sampling for create_clustered_network
    'inhibitory_probability': 0.2,                  # [clustered-only:IGNORED] inhib fraction in create_clustered_network (nc_sim sets E/I via ei_ratio=0.8 internally)
    'within_cluster_prob': 0.3,                     # [clustered-only:IGNORED] within-cluster connection prob (create_clustered_network)
    'between_cluster_prob': 0.15,                   # [clustered-only:IGNORED] between-cluster connection prob (create_clustered_network)
    'target_freq': 10,                              # [always] spike resampling frequency (Hz) for saved data -- NOT a clustered-builder param; used regardless of topology

    # --- hub connections: clustered-builder-only, IGNORED when network_source='nc_sim' ---
    'hub_fraction': 0.1,                            # [clustered-only:IGNORED] nc_sim networks have no hubs
    'hub_between_prob': 0.25,                       # [clustered-only:IGNORED]
    'hub_weight_scale': 1.5,                        # [clustered-only:IGNORED]
    'hub_reciprocal_factor': 2.0,                   # [clustered-only:IGNORED]
}

execute_main_simulation = True
n_sessions_to_generate = 1
session_source = 'latest'
recording_index = 0

In [ ]:
if execute_main_simulation:
    workflow_parameters = inspect.signature(sequential_simulation_individual_saves).parameters
    for session_idx in range(n_sessions_to_generate):
        np.random.seed(seed + session_idx)
        random.seed(seed + session_idx)
        run_params = dict(simulation_params)
        run_params['spontaneous_baseline_seed'] = seed + session_idx
        unsupported_params = sorted(set(run_params) - set(workflow_parameters))
        if unsupported_params:
            raise TypeError(
                'Loaded sequential_simulation_individual_saves() does not support '
                f'{unsupported_params}. Rerun Cell 2 to reload lif_simulation from disk.'
            )
        session_metadata = sequential_simulation_individual_saves(**run_params)
        print(f"Saved session to: {session_metadata['session_dir']}")
else:
    print('Main simulation skipped. Set execute_main_simulation = True to run it.')

In [ ]:
bundle = load_session_bundle(session_source=session_source, recording_index=recording_index)
print(f"Loaded session: {bundle['timestamp']}  ({bundle['relative_session_path']})")

activity = 'stimulus-driven' if bundle['stimulation_enabled'] else 'spontaneous firing'
plot_raster(
    bundle['spike_data_dict'],
    bundle['cluster_assignments'],
    bundle['recording_duration'],
    title=f"nc_sim Recording Raster ({bundle['recording_duration'] / 1000:.0f}s, {activity})",
)
plt.show()

plot_firing_rates(bundle['spike_data_dict'], bundle['cluster_assignments'], bundle['recording_duration'])
plt.show()